# GNN fixed-architecture runs (one operator, seven feature configurations)

Runs on a Kaggle GPU notebook: one operator, the seven feature configurations, the fixed hyperparameters (`conf/hyperparameters/gnn/fixed.yaml`).
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the dataset `sandrokhizanishvili/hi-small-gnn` (graphs + feature_meta.json).
- Set `MODEL` below, then Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/<model>/<run>/seed-42/`.

In [1]:
MODEL = 'gin'            # gin | pna | gat | transformer
FEATURE_CONFIGS = 'c1,c2,c3,c4,c5,c6,c7'
SEED = 42

In [2]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 40.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 51.3 MB/s  0:00:10
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 85.1 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 122.4 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/5

In [3]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn

Cloning into '/kaggle/working/gnn_for_aml'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 96 (delta 21), reused 88 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 62.83 KiB | 5.71 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/kaggle/working/gnn_for_aml
account_to_idx.pkl  HI-Small_accounts.csv  node_features.csv	train_graph.pt
edge_features.csv   HI-Small_Patterns.txt  standard_scaler.pkl	val_graph.pt
feature_meta.json   HI-Small_Trans.csv	   test_graph.pt


In [4]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

python 3.13.15 | torch 2.8.0+cu126 | CUDA build 12.6 | device cuda
  torch_geometric 2.8.1
  torch_scatter 2.1.2+pt28cu126
  torch_sparse 0.6.18+pt28cu126
  lightgbm 4.6.0
  hydra-core 1.3.2
  hydra-submitit-launcher not installed (only needed on Cineca)
  numpy 2.1.3
  scipy 1.16.3
  pandas 2.3.3
  scikit-learn 1.6.1
  matplotlib 3.10.0
  psutil 5.9.5
/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
  gin         params   54,083 | loss 0.7151
  pna         params  176,833 | loss 0.6605
  gat         params   53,953 | loss 0.7310
  transformer params   70,337 | loss 0.7048
OK


In [5]:
# The seven runs, one after another. A finished run is skipped, so the cell can be rerun after an interruption.
!python run.py -m model={MODEL} feature_config={FEATURE_CONFIGS} seed={SEED} machine=kaggle

[2026-10-30 15:04:47,394][HYDRA] Launching 7 jobs locally
[2026-10-30 15:04:47,394][HYDRA] 	#0 : model=gin feature_config=c1 seed=42 machine=kaggle
=== train gin_mp-none_readout-base_dir-in seed 42 -> /kaggle/working/outputs/train/gin/gin_mp-none_readout-base_dir-in/seed-42
mode: train
seed: 42
trial: null
node_encoding: none
gfp_sheet: default
hyperparameters:
  for: gnn
  hidden: 128
  num_layers: 2
  dropout: 0.3
  norm: none
  residual: identity
  ffn: false
  num_neighbors:
  - 100
  - 100
  batch_size: 8192
  epochs: 20
  lr: 0.001
  weight_decay: 1.0e-05
  pos_weight: 8.0
  self_loops: true
  readout_layers: 2
  readout_width: 1.0
  lr_schedule: cosine
  num_workers: 0
  select_metric: pr_auc
model:
  name: gin
  kind: gnn
feature_config:
  id: 1
  mp: none
  readout: base
machine: kaggle
paths:
  data: /kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn
  outputs: /kaggle/working/outputs

edge_attr width 81 (base 20 + gfp 61, sheet default)
train nodes=515,070 edges=3,046,3

In [6]:
# The metrics of every finished run of this operator, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(f'/kaggle/working/outputs/train/{MODEL}/*/seed-*/results.json'))]
cols = ['run', 'best_epoch', 'threshold', 'params', 'invariant_params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

,run,best_epoch,threshold,params,invariant_params,train_f1,train_precision,train_recall,train_pr_auc,val_f1,val_precision,val_recall,val_pr_auc,test_f1,test_precision,test_recall,test_pr_auc
0,gin_mp-base_readout-base_dir-in,19,0.6299,108419,67587,0.5780,0.7782,0.4597,0.5328,0.5583,0.8691,0.4113,0.4951,0.4743,0.7318,0.3508,0.4182
1,gin_mp-base_readout-full_dir-in,16,0.6768,116227,67587,0.5685,0.7239,0.4680,0.5363,0.6066,0.8169,0.4824,0.5603,0.5148,0.6481,0.4269,0.4768
2,gin_mp-base_readout-gfp_dir-in,18,0.7423,113667,67587,0.5313,0.7788,0.4031,0.4978,0.5265,0.7404,0.4085,0.4616,0.4568,0.5900,0.3727,0.3990
3,gin_mp-full_readout-full_dir-in,10,0.6418,131843,67587,0.5752,0.7806,0.4554,0.5504,0.6099,0.8671,0.4704,0.5549,0.5295,0.7553,0.4077,0.4922
4,gin_mp-none_readout-base_dir-in,17,0.4606,103043,67587,0.4222,0.4819,0.3757,0.3836,0.4354,0.6398,0.3299,0.3761,0.3699,0.4195,0.3307,0.3369
5,gin_mp-none_readout-full_dir-in,11,0.6056,110851,67587,0.4620,0.6125,0.3709,0.4149,0.5509,0.8193,0.4150,0.4890,0.4817,0.6751,0.3745,0.4432
6,gin_mp-none_readout-gfp_dir-in,16,0.6609,108291,67587,0.2718,0.6131,0.1746,0.2373,0.3219,0.4310,0.2569,0.2383,0.2576,0.2545,0.2607,0.2043


In [7]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

-rw-r--r-- 1 root root 1.4M Oct 30 18:28 results_small.zip


## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`, about 1.6 GB for seven runs).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the rows in `EXPERIMENTS.md`.
- Next operator: change `MODEL` and run again in a new session.